# DVC jako „Git dla danych i modeli”

Cel: pokazać minimalny workflow wersjonowania danych i modelu. Ten notebook jest przygotowany do screencastu: część komend wykonuj w terminalu, ponieważ DVC najlepiej pokazuje się jako narzędzie CLI.


## Instalacja

```bash
pip install dvc scikit-learn pandas joblib
```

W repozytorium projektu uruchom:

```bash
git init
dvc init
```


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import joblib

Path("data").mkdir(exist_ok=True)
Path("models").mkdir(exist_ok=True)

iris = load_iris(as_frame=True)
df = iris.frame
df.to_csv("data/iris.csv", index=False)
print(df.head())


## Dodanie danych do DVC

W terminalu:

```bash
dvc add data/iris.csv
git add data/iris.csv.dvc data/.gitignore .dvc .dvcignore
git commit -m "Add Iris dataset tracked by DVC"
```

Opcjonalny lokalny remote:

```bash
mkdir -p /tmp/dvc-storage
dvc remote add -d localstorage /tmp/dvc-storage
dvc push
```


In [ ]:
df = pd.read_csv("data/iris.csv")
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=50, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
acc = accuracy_score(y_test, preds)
print({"accuracy": acc})

joblib.dump({"model": model, "accuracy": acc, "version": "v1"}, "models/iris_rf.joblib")


## Wersjonowanie modelu

W terminalu:

```bash
dvc add models/iris_rf.joblib
git add models/iris_rf.joblib.dvc models/.gitignore
git commit -m "Add first RF model version"
dvc push
```

Następnie zmień parametry modelu, np. `n_estimators=200`, ponownie wytrenuj model i wykonaj:

```bash
dvc add models/iris_rf.joblib
git add models/iris_rf.joblib.dvc
git commit -m "Update RF model version"
dvc push
```


## Co powiedzieć podczas screencastu

- Git trzyma kod i lekkie metadane DVC.
- DVC trzyma duże dane i modele w storage.
- Wersja modelu bez wersji danych jest niepełna.
- Możemy wrócić do starego commita i wykonać `dvc pull`, żeby odtworzyć dokładny stan danych/modelu.
